# Step 0.2 — Crop masks from MIRCA2000, seuil pixel=2% / cellule=2% (raisonnement agronomique)

**Choix agronomique (pas base sur la correlation GAEZ)**, discute avec l'utilisatrice le 2026-10-13 --
voir [[mirca-5pct-presence-threshold-dilution]] et [[soil-suitability-module-iv-fixes]] pour le
contexte complet :

- **Seuil pixel natif (5 arc-min, ~2%)** : filtre le vrai bruit (jardin, essai isole, <<1% de part)
  sans exiger une dominance locale irrealiste -- l'agriculture europeenne est en rotation/diversifiee,
  pas monoculturale. 2% d'un pixel natif (~140 ha sur un pixel typique de ~7000 ha) represente deja
  quelques champs reels, pas une trace.
- **Seuil cellule 1deg (2%)** : verifie empiriquement (sur Maize) que meme au seuil le plus bas teste,
  les cellules nouvellement ajoutees ont typiquement 3-7 pixels natifs concernes (median 4 sur 52
  cellules testees), pas 1 seul pixel isole -- le risque de "poche de microclimat isolee non
  representative du climat moyen de la cellule" (qui aurait justifie un seuil cellule plus strict)
  ne se materialise pas vraiment dans les donnees reelles.

Parallele, non-destructif : construit `mirca2000_1deg_threshold2pct_<Crop>_rf.nc`, rien dans
`mirca2000_1deg_*.nc` (seuil original 5%/5%) ni `mirca2000_no_threshold_1deg_*.nc` (0%/0%, la
branche precedente) n'est touche.


In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np
import pandas as pd
import xarray as xr
import regionmask
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
CAL_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "growing_periods_listed" / "CELL_SPECIFIC_CROPPING_CALENDARS.TXT.gz"
CELL_AREA_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "cell_area_grid" / "cell_area_ha.npy"
ERA5_1DEG = BASE / "1_SVM" / "0_preprocessing_masks" / "outputs_input_data_1deg"

PIXEL_THR = 0.02  # part de surface d'un pixel natif pour compter comme "cropped"
CELL_THR  = 0.02  # part de pixels natifs "cropped" dans la cellule pour compter comme "presente"

LAT0, LAT1, LON0, LON1 = 35, 71, -25, 44
NLAT, NLON = LAT1 - LAT0 + 1, LON1 - LON0 + 1
NCELL = NLAT * NLON
LAND1_THR = 0.25
lat1 = np.arange(LAT0, LAT1 + 1); lon1 = np.arange(LON0, LON1 + 1)
print(f"1deg grid: {NLAT} x {NLON} = {NCELL} cells, PIXEL_THR={PIXEL_THR}, CELL_THR={CELL_THR}")

CROP_MAP = {
    'Maize_rf':        ('Maize',  'rf', None,     'mirca2000_1deg_Maize_rf.nc'),
    'Wheat-spring_rf': ('Wheat',  'rf', 'spring', 'mirca2000_1deg_Wheat-spring_rf.nc'),
    'Wheat-winter_rf': ('Wheat',  'rf', 'winter', 'mirca2000_1deg_Wheat-winter_rf.nc'),
    'Potato_rf':       ('Potato', 'rf', None,     'mirca2000_1deg_Potato_rf.nc'),
}


1deg grid: 37 x 70 = 2590 cells, PIXEL_THR=0.02, CELL_THR=0.02


C:\Users\elisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Grille cible + masque terre (meme source que 0.1/0.4)

In [2]:
sample_file = ERA5_1DEG / "ERA5_1deg_tasmax_1981_2015_daily.nc"
ds_grid = xr.open_dataset(sample_file)
land_mask = np.isfinite(ds_grid['tasmax'].isel(time=0).values)
ds_grid.close()
print(f"Land 1deg cells: {int(land_mask.sum())} / {land_mask.size}")


Land 1deg cells: 1332 / 2590


## 2. MIRCA2000 presence a 1deg, seuil pixel=2% / cellule=2%

In [3]:
NROW, NCOL = 2160, 4320
ROWS_, COLS_ = np.arange(NROW), np.arange(NCOL)
lat5 = 90 - (ROWS_ + 0.5) / 12
lon5 = -180 + (COLS_ + 0.5) / 12
lat5_i = np.floor(lat5 + 0.5).astype(int)
lon5_i = np.floor(lon5 + 0.5).astype(int)
R5 = np.where((lat5_i >= LAT0) & (lat5_i <= LAT1))[0]
C5 = np.where((lon5_i >= LON0) & (lon5_i <= LON1))[0]
row_map = {r: i for i, r in enumerate(R5)}
col_map = {c: j for j, c in enumerate(C5)}
CELL5 = (lat5_i[R5] - LAT0)[:, None] * NLON + (lon5_i[C5] - LON0)[None, :]
AREA5 = np.load(CELL_AREA_FILE)[np.ix_(R5, C5)]

# Masque terre natif + masque terre 1deg partage (meme logique que 0.1), necessaire pour le
# denominateur "n_keep" du seuil cellule (fraction de pixels NATIFS-TERRE "cropped")
land_rg = regionmask.defined_regions.natural_earth_v5_0_0.land_10
land5 = np.isfinite(land_rg.mask(lon5[C5], lat5[R5]).values)
land_frac = np.bincount(CELL5.ravel(), weights=land5.ravel().astype(float), minlength=NCELL) / 144.0
LAND1 = land_frac >= LAND1_THR
keep5 = land5 & LAND1[CELL5]
n_keep = np.bincount(CELL5.ravel(), weights=keep5.ravel().astype(float), minlength=NCELL).reshape(NLAT, NLON)

CAL_CLASSES = {"Wheat": (1, 27), "Maize": (2, 28), "Potato": (10, 36)}
cols = ["cell", "row", "col", "lat", "lon", "crop", "subcrop", "area", "start", "end"]
cal = pd.read_csv(CAL_FILE, sep="\t", compression="gzip", usecols=range(10), header=0, names=cols)
allc = [c for v in CAL_CLASSES.values() for c in v]
cal = cal[cal.crop.isin(allc)].copy()
cal["i"] = (cal.row - 1).map(row_map)
cal["j"] = (cal.col - 1).map(col_map)
cal = cal.dropna(subset=["i", "j"])
cal["i"] = cal.i.astype(int); cal["j"] = cal.j.astype(int)
cal["irr"] = np.where(cal.crop <= 26, "ir", "rf")
name_of = {c: k for k, v in CAL_CLASSES.items() for c in v}
cal["crop_name"] = cal.crop.map(name_of)
print("calendar sub-crop rows in window:", len(cal))


def mirca2000_presence_1deg_threshold2pct(crop_name, irr, season=None):
    g = cal[(cal.crop_name == crop_name) & (cal.irr == irr)]
    if season == 'winter':
        g = g[g.start.isin([9, 10, 11, 12])]
    elif season == 'spring':
        g = g[g.start.isin([2, 3, 4, 5])]
    ii, jj, ar = g.i.values, g.j.values, g.area.values
    A = np.zeros((len(R5), len(C5)))
    np.add.at(A, (ii, jj), ar)
    share = A / AREA5
    cropped = keep5 & (share > PIXEL_THR)                                   # seuil pixel
    n_crop = np.bincount(CELL5.ravel(), weights=cropped.ravel().astype(float), minlength=NCELL).reshape(NLAT, NLON)
    cell_frac = np.where(n_keep > 0, n_crop / np.maximum(n_keep, 1), 0.0)
    pres = (n_keep > 0) & (cell_frac > CELL_THR)                            # seuil cellule
    area_cell = np.bincount(CELL5.ravel(), weights=A.ravel(), minlength=NCELL).reshape(NLAT, NLON)
    return np.where(land_mask, pres.astype(float), np.nan), np.where(land_mask, area_cell, np.nan)


presence_by_label = {}
for crop_label, (crop_name, irr, season, _old) in CROP_MAP.items():
    pres, area_cell = mirca2000_presence_1deg_threshold2pct(crop_name, irr, season)
    presence_by_label[crop_label] = pres
    n_pres = int(np.nansum(pres == 1)); n_land = int(np.sum(land_mask))
    print(f"{crop_label:18s}: presence (pixel>{PIXEL_THR*100:.0f}%, cellule>{CELL_THR*100:.0f}%) = "
          f"{n_pres} / {n_land} land cells ({100*n_pres/n_land:.1f}%)")


calendar sub-crop rows in window: 591065
Maize_rf          : presence (pixel>2%, cellule>2%) = 617 / 1332 land cells (46.3%)
Wheat-spring_rf   : presence (pixel>2%, cellule>2%) = 255 / 1332 land cells (19.1%)
Wheat-winter_rf   : presence (pixel>2%, cellule>2%) = 887 / 1332 land cells (66.6%)


Potato_rf         : presence (pixel>2%, cellule>2%) = 405 / 1332 land cells (30.4%)


## 3. Sauvegarde -- reutilise le calendrier (sowing/maturity DOY) de MIRCA2000 original, inchange

In [4]:
for crop_label, (crop_name, irr, season, old_file) in CROP_MAP.items():
    old_ds = xr.open_dataset(ERA5_1DEG / old_file)
    sowing_doy = old_ds['sowing_doy'].values
    maturity_doy = old_ds['maturity_doy'].values
    lats_ref = old_ds.lat.values; lons_ref = old_ds.lon.values
    old_ds.close()

    new_presence = presence_by_label[crop_label]
    out = xr.Dataset({
        'presence': xr.DataArray(new_presence, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                 attrs={'long_name': f'MIRCA2000 presence, pixel>{PIXEL_THR*100:.0f}% cellule>{CELL_THR*100:.0f}%, crop={crop_label}'}),
        'sowing_doy': xr.DataArray(sowing_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                   attrs={'long_name': 'Sowing DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
        'maturity_doy': xr.DataArray(maturity_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                     attrs={'long_name': 'Maturity DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
    })
    out.attrs['description'] = (f'Presence from MIRCA2000, pixel>{PIXEL_THR} cellule>{CELL_THR} (agronomic choice, not GAEZ-tuned) -- {crop_label}')
    out_path = ERA5_1DEG / f'mirca2000_threshold2pct_1deg_{crop_label}.nc'
    out.to_netcdf(out_path)
    print(f"Saved: {out_path.name}  (presence cells: {int(np.nansum(new_presence==1))})")


Saved: mirca2000_threshold2pct_1deg_Maize_rf.nc  (presence cells: 617)


Saved: mirca2000_threshold2pct_1deg_Wheat-spring_rf.nc  (presence cells: 255)


Saved: mirca2000_threshold2pct_1deg_Wheat-winter_rf.nc  (presence cells: 887)
Saved: mirca2000_threshold2pct_1deg_Potato_rf.nc  (presence cells: 405)
